In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("All libraries imported successfully!")

All libraries imported successfully!


In [22]:
df = pd.read_csv("student_performance.csv")
print("Dataset loaded. Total rows and columns:", df.shape)

Dataset loaded. Total rows and columns: (100, 8)


In [23]:
df.columns = df.columns.str.strip().str.replace(" ", "_").str.title()

print("Column names now:")
print(df.columns.tolist())

Column names now:
['Student_Id', 'Gender', 'Study_Hours', 'Attendance_Percent', 'Sleep_Hours', 'Previous_Marks', 'Internet_Access', 'Final_Marks']


In [24]:
df = df.rename(columns={"Result": "Final_Result"})

In [25]:
df.head()

,Student_Id,Gender,Study_Hours,Attendance_Percent,Sleep_Hours,Previous_Marks,Internet_Access,Final_Marks
0,S001,Female,7.2,71.0,8.0,56.0,Yes,68.0
1,S002,Female,4.5,NaN,8.9,84.0,Yes,67.0
2,S003,Female,7.9,77.0,6.8,35.0,Yes,55.0
3,S004,Male,6.6,86.0,5.1,88.0,Yes,80.0
4,S005,Female,NaN,75.0,4.6,67.0,Yes,45.0


In [26]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
Student_Id            0
Gender                4
Study_Hours           8
Attendance_Percent    6
Sleep_Hours           7
Previous_Marks        5
Internet_Access       3
Final_Marks           0
dtype: int64


In [28]:
# Keep only the 4 features and Final_Marks (needed to make the target)
df = df[["Study_Hours", "Attendance_Percent", "Previous_Marks", "Sleep_Hours", "Final_Marks"]].copy()

# Use the same column name as the assignment
df = df.rename(columns={"Study_Hours": "Hours_Studied"})

# Pass if Final_Marks is 40 or more, otherwise Fail
pass_mark = 40
df["Final_Result"] = df["Final_Marks"].apply(lambda m: "Pass" if m >= pass_mark else "Fail")

# Final_Marks must be removed, otherwise the model would get the answer directly
df = df.drop("Final_Marks", axis=1)

print("Columns now:", df.columns.tolist())
print()
print(df["Final_Result"].value_counts())

Columns now: ['Hours_Studied', 'Attendance_Percent', 'Previous_Marks', 'Sleep_Hours', 'Final_Result']

Final_Result
Pass    95
Fail     5
Name: count, dtype: int64


In [29]:
df.head()

,Hours_Studied,Attendance_Percent,Previous_Marks,Sleep_Hours,Final_Result
0,7.2,71.0,56.0,8.0,Pass
1,4.5,NaN,84.0,8.9,Pass
2,7.9,77.0,35.0,6.8,Pass
3,6.6,86.0,88.0,5.1,Pass
4,NaN,75.0,67.0,4.6,Pass


In [30]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
Hours_Studied         8
Attendance_Percent    6
Previous_Marks        5
Sleep_Hours           7
Final_Result          0
dtype: int64


In [31]:
X = df.drop("Final_Result", axis=1)   # all columns except Final_Result
y = df["Final_Result"]                # only the Final_Result column

print("Shape of X (features):", X.shape)
print("Shape of y (target)  :", y.shape)
print()
print("Feature columns:", list(X.columns))
print("Target column  :", y.name)

Shape of X (features): (100, 4)
Shape of y (target)  : (100,)

Feature columns: ['Hours_Studied', 'Attendance_Percent', 'Previous_Marks', 'Sleep_Hours']
Target column  : Final_Result


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% training, 20% testing
    random_state=42,    # for reproducible results
    stratify=y          # keep the Pass/Fail ratio same
)

print("Split done successfully!")

Split done successfully!


In [33]:
print("Full data     (%):")
print((y.value_counts(normalize=True) * 100).round(1))
print()
print("Training data (%):")
print((y_train.value_counts(normalize=True) * 100).round(1))
print()
print("Testing data  (%):")
print((y_test.value_counts(normalize=True) * 100).round(1))

Full data     (%):
Final_Result
Pass    95.0
Fail     5.0
Name: proportion, dtype: float64

Training data (%):
Final_Result
Pass    95.0
Fail     5.0
Name: proportion, dtype: float64

Testing data  (%):
Final_Result
Pass    95.0
Fail     5.0
Name: proportion, dtype: float64


In [34]:
imputer = SimpleImputer(strategy="median")

# Learn medians from TRAINING data only, then fill training data
X_train_imputed = imputer.fit_transform(X_train)

# Use the SAME training medians to fill testing data (no fit here!)
X_test_imputed = imputer.transform(X_test)

print("Medians learned from training data:")
for col, med in zip(X.columns, imputer.statistics_):
    print(f"  {col:20s} -> {med}")

print()
print("Missing values left in X_train:", pd.DataFrame(X_train_imputed).isnull().sum().sum())
print("Missing values left in X_test :", pd.DataFrame(X_test_imputed).isnull().sum().sum())

Medians learned from training data:
  Hours_Studied        -> 4.8
  Attendance_Percent   -> 76.0
  Previous_Marks       -> 63.0
  Sleep_Hours          -> 6.85

Missing values left in X_train: 0
Missing values left in X_test : 0


In [35]:
scaler = StandardScaler()

# Learn mean and std from TRAINING data only
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Apply the SAME training mean and std to testing data (no fit here!)
X_test_scaled = scaler.transform(X_test_imputed)

print("Mean learned from training data:")
print(pd.Series(scaler.mean_, index=X.columns).round(2))
print()
print("Standard deviation learned from training data:")
print(pd.Series(scaler.scale_, index=X.columns).round(2))

Mean learned from training data:
Hours_Studied          4.96
Attendance_Percent    76.01
Previous_Marks        64.24
Sleep_Hours            6.67
dtype: float64

Standard deviation learned from training data:
Hours_Studied          2.13
Attendance_Percent    11.89
Previous_Marks        17.56
Sleep_Hours            1.28
dtype: float64


In [36]:
X_train_final = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_final  = pd.DataFrame(X_test_scaled,  columns=X.columns, index=X_test.index)

print("Training data after scaling - mean (should be ~0):")
print(X_train_final.mean().round(2) + 0)
print()
print("Training data after scaling - std (should be ~1):")
print(X_train_final.std(ddof=0).round(2))

Training data after scaling - mean (should be ~0):
Hours_Studied         0.0
Attendance_Percent    0.0
Previous_Marks        0.0
Sleep_Hours           0.0
dtype: float64

Training data after scaling - std (should be ~1):
Hours_Studied         1.0
Attendance_Percent    1.0
Previous_Marks        1.0
Sleep_Hours           1.0
dtype: float64


In [37]:
print("Shape of X_train:", X_train_final.shape)
print("Shape of X_test :", X_test_final.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test :", y_test.shape)

Shape of X_train: (80, 4)
Shape of X_test : (20, 4)
Shape of y_train: (80,)
Shape of y_test : (20,)


In [38]:
X_train_final.head()

,Hours_Studied,Attendance_Percent,Previous_Marks,Sleep_Hours
4,-0.073334,-0.085143,0.157336,-1.617786
16,0.208269,-1.514713,0.783834,-1.226780
19,0.536806,1.596703,0.954697,-0.288367
33,-0.448805,1.092149,1.125559,0.415444
25,-1.105878,-0.673790,-0.924796,1.588461


In [39]:
X_test_final.head()

,Hours_Studied,Attendance_Percent,Previous_Marks,Sleep_Hours
92,-0.167202,1.764888,-0.412207,1.588461
91,-0.073334,-0.757882,-1.095658,-1.539585
2,1.381614,0.083041,-1.665201,0.102639
20,1.006144,-1.346528,-0.583070,-0.601171
38,-1.012010,-0.673790,-0.981750,0.141740
